In [1]:
import os
import xml.etree.ElementTree as ET

In [ ]:
ROOT = #your root path here

ANNOTATIONS_DIR = os.path.join(ROOT, "annotations")

IR_DIRS = {
    "train": os.path.join(ROOT, "images", "train"),
    "val": os.path.join(ROOT, "images", "val"),
}

LABEL_DIRS = {
    "train": os.path.join(ROOT, "labels", "train"),
    "val": os.path.join(ROOT, "labels", "val"),
}

CLASS_MAP = {
    "person": 0
}

In [ ]:
def convert_box(size, box):
    width, height = size

    xmin, ymin, xmax, ymax = box

    x_center = ((xmin + xmax) / 2) / width
    y_center = ((ymin + ymax) / 2) / height
    box_width = (xmax - xmin) / width
    box_height = (ymax - ymin) / height

    return x_center, y_center, box_width, box_height

In [ ]:
for split in ["train", "val"]:

    os.makedirs(LABEL_DIRS[split], exist_ok=True)

    ir_names = os.listdir(IR_DIRS[split])

    for irn in ir_names:

        xml_name = irn.replace(".jpg", ".xml")
        xml_path = os.path.join(ANNOTATIONS_DIR, xml_name)

        if not os.path.exists(xml_path):
            print(f"Missing annotation: {xml_name}")
            continue

        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find("size")
        width = int(size.find("width").text)
        height = int(size.find("height").text)

        txt_name = irn.replace(".jpg", ".txt")
        txt_path = os.path.join(LABEL_DIRS[split], txt_name)

        with open(txt_path, "w") as f:

            for obj in root.findall("object"):

                c = obj.find("name").text

                if c not in CLASS_MAP:
                    continue

                c_id = CLASS_MAP[c]

                box = obj.find("bndbox")

                xmin = float(box.find("xmin").text)
                ymin = float(box.find("ymin").text)
                xmax = float(box.find("xmax").text)
                ymax = float(box.find("ymax").text)

                x, y, w, h = convert_box((width, height), (xmin, ymin, xmax, ymax)
                )

                f.write(f"{c_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

print("Conversion Complete")

Conversion Complete
